# NB-04_salida_cg_coy

Process Flow SAS: **Salida CG coy** — `ProcessFlowContainer-8iloVuuXSe8CJiAb`

## OPC Agregar IMPUT a permanentes*

Actualiza las imputaciones permanentes de gasto (agregado y desagregado por partida) reemplazando las combinaciones ya existentes con las recién calculadas del periodo

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE EXISTS correlacionado + PROC DATASETS APPEND FORCE + DROP TABLE*

In [ ]:
# ========= OPC Agregar IMPUT a permanentes* =========
# /*ACTUALIZACION DE LAS IMPUTACIONES PERMANENTES (Sólo si corresponden a permanentes!) */
# /* para salida de indicadores*/
# EJECUCION_CG_DIP_imputaciones (WORK) queda materializada como #tmp por el nodo anterior (session_sql)
sql_delete_imput_gc_dip = """
DELETE FROM GOBGENER.dbo.IMPUT_GC_DIP
WHERE EXISTS (
    SELECT 1 FROM #ejecucion_cg_dip_imputaciones e
    WHERE e.SUBSECTOR = IMPUT_GC_DIP.SUBSECTOR AND e.MONEDA = IMPUT_GC_DIP.MONEDA
    AND e.AÑO = IMPUT_GC_DIP.AÑO AND e.TRIM = IMPUT_GC_DIP.TRIM
    AND e.MES = IMPUT_GC_DIP.MES AND e.C_TIPO_CUENTA = IMPUT_GC_DIP.C_TIPO_CUENTA
    AND e.C_CUENTA = IMPUT_GC_DIP.C_CUENTA AND e.C_ITEM = IMPUT_GC_DIP.C_ITEM
    AND e.C_SCN = IMPUT_GC_DIP.C_SCN AND e.C_CAGENTE = IMPUT_GC_DIP.C_CAGENTE
    AND e.C_TRANSACCION = IMPUT_GC_DIP.C_TRANSACCION
    AND e.T_TRANSACCION = IMPUT_GC_DIP.T_TRANSACCION AND
    e.C_ACTIVO = IMPUT_GC_DIP.C_ACTIVO AND e.C_CUP = IMPUT_GC_DIP.C_CUP
    AND e.C_ACT = IMPUT_GC_DIP.C_ACT AND e.[KEY] = IMPUT_GC_DIP.[KEY]
    AND e.NIVEL = IMPUT_GC_DIP.NIVEL
)
"""
res = work_conn.execute(text(sql_delete_imput_gc_dip))
_log("DELETE GOBGENER.dbo.IMPUT_GC_DIP", res.rowcount)


In [ ]:
# PROC DATASETS APPEND FORCE: inserta TODAS las columnas del dataset origen
# (FORCE tolera diferencias de estructura), no solo las de la clave del EXISTS.
cols_ejecucion_cg_dip_imputaciones = pd.read_sql(
    text("SELECT TOP 0 * FROM #ejecucion_cg_dip_imputaciones"), work_conn
).columns.tolist()
cols_sql_imput_gc_dip = ", ".join(f"[{c}]" for c in cols_ejecucion_cg_dip_imputaciones)
sql_append_imput_gc_dip = f"""
INSERT INTO GOBGENER.dbo.IMPUT_GC_DIP ({cols_sql_imput_gc_dip})
SELECT {cols_sql_imput_gc_dip}
FROM #ejecucion_cg_dip_imputaciones
"""
res = work_conn.execute(text(sql_append_imput_gc_dip))
_log("APPEND GOBGENER.dbo.IMPUT_GC_DIP", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imputaciones"))


In [ ]:
# /* para análisis desagregado*/
# EJECUCION_CG_DIP_APER_IMP (WORK) queda materializada como #tmp por el nodo anterior (session_sql)
sql_delete_imput_gc_dip_aper = """
DELETE FROM GOBGENER.dbo.IMPUT_GC_DIP_APER
WHERE EXISTS (
    SELECT 1 FROM #ejecucion_cg_dip_aper_imp e
    WHERE e.SUBSECTOR = IMPUT_GC_DIP_APER.SUBSECTOR AND e.MONEDA = IMPUT_GC_DIP_APER.MONEDA
    AND e.AÑO = IMPUT_GC_DIP_APER.AÑO AND e.TRIM = IMPUT_GC_DIP_APER.TRIM
    AND e.MES = IMPUT_GC_DIP_APER.MES AND e.C_ENTIDAD = IMPUT_GC_DIP_APER.C_ENTIDAD
    AND e.C_PARTIDA = IMPUT_GC_DIP_APER.C_PARTIDA AND e.C_CAPITULO = IMPUT_GC_DIP_APER.C_CAPITULO
    AND e.C_PROGRAMA = IMPUT_GC_DIP_APER.C_PROGRAMA AND e.C_TIPO_CUENTA = IMPUT_GC_DIP_APER.C_TIPO_CUENTA
    AND e.C_CUENTA = IMPUT_GC_DIP_APER.C_CUENTA AND e.C_ITEM = IMPUT_GC_DIP_APER.C_ITEM AND
    e.C_ASIGNACION = IMPUT_GC_DIP_APER.C_ASIGNACION AND e.C_ANALITICO = IMPUT_GC_DIP_APER.C_ANALITICO
    AND e.N_ANALITICO = IMPUT_GC_DIP_APER.N_ANALITICO AND e.C_SCN = IMPUT_GC_DIP_APER.C_SCN
    AND e.C_CAGENTE = IMPUT_GC_DIP_APER.C_CAGENTE AND e.C_TRANSACCION = IMPUT_GC_DIP_APER.C_TRANSACCION
    AND e.T_TRANSACCION = IMPUT_GC_DIP_APER.T_TRANSACCION AND e.C_ACTIVO = IMPUT_GC_DIP_APER.C_ACTIVO
    AND e.C_CUP = IMPUT_GC_DIP_APER.C_CUP AND e.C_ACT = IMPUT_GC_DIP_APER.C_ACT AND e.NIVEL = IMPUT_GC_DIP_APER.NIVEL
)
"""
res = work_conn.execute(text(sql_delete_imput_gc_dip_aper))
_log("DELETE GOBGENER.dbo.IMPUT_GC_DIP_APER", res.rowcount)


In [ ]:
# PROC DATASETS APPEND FORCE: inserta TODAS las columnas del dataset origen
# (FORCE tolera diferencias de estructura), no solo las de la clave del EXISTS.
cols_ejecucion_cg_dip_aper_imp = pd.read_sql(
    text("SELECT TOP 0 * FROM #ejecucion_cg_dip_aper_imp"), work_conn
).columns.tolist()
cols_sql_imput_gc_dip_aper = ", ".join(f"[{c}]" for c in cols_ejecucion_cg_dip_aper_imp)
sql_append_imput_gc_dip_aper = f"""
INSERT INTO GOBGENER.dbo.IMPUT_GC_DIP_APER ({cols_sql_imput_gc_dip_aper})
SELECT {cols_sql_imput_gc_dip_aper}
FROM #ejecucion_cg_dip_aper_imp
"""
res = work_conn.execute(text(sql_append_imput_gc_dip_aper))
_log("APPEND GOBGENER.dbo.IMPUT_GC_DIP_APER", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))


## IMPUT DIP Aper_CG_coy

Ejecución DIPRES desagregada por entidad imputando analíticos predefinidos, variables y transacciones diferenciadas por coyuntura/actividad, consolidada en una única tabla marcada como Imputación

*confianza: medium · verificador: revise · SAS: PROC SQL: LEFT JOINs de imputación con GROUP BY parcial (CASE + agregación tipo 'primer registro'), múltiples DELETE, UNION ALL y UPDATE final*

In [ ]:
# ========= IMPUT DIP Aper_CG_coy =========
# /* IMPUTACIONES: Salida para ANALISIS DESAGREGADO de indicadores GC de consumo de gobierno (coyuntura) */
# /*MODIFICADO JUNIO 2025*/
# Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_y"))
sql_ejec_dip_y = f"""
SELECT T1.*, T2.C_ENTIDAD, t2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip_y
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
  ON (T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA)
WHERE t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip_y))
ejec_dip_y = pd.read_sql(text("SELECT * FROM #ejec_dip_y"), work_conn)
_log("ejec_dip_y", ejec_dip_y)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper_imp"))
sql_ejecucion_a_prelim_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON (t1.C_TIPO_CUENTA = t2.T_CUENTA) AND (t1.C_CUENTA = t2.C_CUENTA) AND (t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper_imp))


In [ ]:
# DELETE FROM EJECUCION_A_prelim_DIP_aper_imp WHERE C_ENTIDAD IN (1500040001)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
# DELETE FROM EJECUCION_A_prelim_DIP_aper_imp WHERE C_CUENTA='32' AND C_ANALITICO IN (...)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (cuenta 32)", res.rowcount)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (ajuste CORFO)", res.rowcount)
ejecucion_a_prelim_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_aper_imp"), work_conn)
_log("ejecucion_a_prelim_dip_aper_imp", ejecucion_a_prelim_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos (sigue)*/
# GROUP BY exactamente como en el SAS: no incluye C_PARTIDA, C_CAPITULO, C_PROGRAMA,
# C_ASIGNACION, N_ANALITICO ni DEVENGO aunque están en el SELECT -> PROC SQL toma
# el valor del primer registro del grupo para esas columnas (colapsa filas)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper_imp"))
sql_ejecucion_a_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       MIN(t1.C_PARTIDA) AS C_PARTIDA,
       MIN(t1.C_CAPITULO) AS C_CAPITULO,
       MIN(t1.C_PROGRAMA) AS C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       MIN(t1.C_ASIGNACION) AS C_ASIGNACION,
       t1.C_ANALITICO,
       MIN(t1.N_ANALITICO) AS N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       t1.C_ACT,
       t1.NIVEL,
       MIN(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper_imp
FROM #ejecucion_a_prelim_dip_aper_imp t1
WHERE t1.C_ENTIDAD IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND t1.MONEDA IN ('{Moneda}')
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    t1.C_ACT,
    t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper_imp))
ejecucion_a_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_aper_imp"), work_conn)
_log("ejecucion_a_dip_aper_imp", ejecucion_a_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper_imp"))
sql_ejecucion_b_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper_imp))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_b_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
sql_delete_b = f"""
DELETE FROM #ejecucion_b_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_b))
_log("DELETE #ejecucion_b_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_b_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_aper_imp"), work_conn)
_log("ejecucion_b_dip_aper_imp", ejecucion_b_dip_aper_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper_imp"))
sql_ejecucion_c_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper_imp))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_c_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
sql_delete_c = f"""
DELETE FROM #ejecucion_c_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_c))
_log("DELETE #ejecucion_c_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_c_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_aper_imp"), work_conn)
_log("ejecucion_c_dip_aper_imp", ejecucion_c_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper_imp"))
sql_ejecucion_d_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper_imp))


In [ ]:
sql_delete_d = f"""
DELETE FROM #ejecucion_d_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_d))
_log("DELETE #ejecucion_d_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_d_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_aper_imp"), work_conn)
_log("ejecucion_d_dip_aper_imp", ejecucion_d_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper_imp"))
sql_ejecucion_f_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper_imp))


In [ ]:
sql_delete_f = f"""
DELETE FROM #ejecucion_f_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_f))
_log("DELETE #ejecucion_f_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_f_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_aper_imp"), work_conn)
_log("ejecucion_f_dip_aper_imp", ejecucion_f_dip_aper_imp)


In [ ]:
# EJECUCION_G_DIP_aper_imp (DROP=ACTIVIDAD)
# GROUP BY exactamente como en el SAS: no incluye ACTIVIDAD (solo se usa para
# derivar C_ACT en el SELECT); PROC SQL toma el valor del primer registro del
# grupo para esa columna al calcular C_ACT
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper_imp"))
sql_ejecucion_g_dip_aper_imp = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN MIN(t1.ACTIVIDAD) = 'FFAA' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'FFOO' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'PREVISION' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'FONASA' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'CORFO' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'CENABAST' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'AAPP' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'SSPP' THEN 'SSPP'
             WHEN MIN(t1.ACTIVIDAD) = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.dato
INTO #ejecucion_g_dip_aper_imp
FROM #ejec_z_mixt_aper_dip_imp t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    t1.C_ASIGNACION,
    t1.N_ANALITICO,
    t1.NIVEL,
    t1.dato
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper_imp))
ejecucion_g_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_aper_imp"), work_conn)
_log("ejecucion_g_dip_aper_imp", ejecucion_g_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))
sql_union = """
SELECT * INTO #ejecucion_cg_dip_aper_imp FROM (
  SELECT * FROM #ejecucion_a_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_b_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_c_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_d_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_g_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_f_dip_aper_imp
) u
"""
work_conn.execute(text(sql_union))


In [ ]:
# DROP TABLE de las intermedias (equivalente server-side: se dropean las #tmp)
for _t in ["#ejecucion_a_prelim_dip_aper_imp", "#ejecucion_a_dip_aper_imp", "#ejecucion_b_dip_aper_imp", "#ejecucion_c_dip_aper_imp", "#ejecucion_d_dip_aper_imp", "#ejecucion_g_dip_aper_imp", "#ejecucion_f_dip_aper_imp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# UPDATE EJECUCION_CG_DIP_aper_imp SET N_CUENTA='Imputacion'
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_imp SET N_CUENTA = 'Imputacion'"))
_log("UPDATE #ejecucion_cg_dip_aper_imp (N_CUENTA=Imputacion)", res.rowcount)
ejecucion_cg_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_imp"), work_conn)
_log("ejecucion_cg_dip_aper_imp", ejecucion_cg_dip_aper_imp)


## OPC Salida CG_coy (solo DIP)

Arma la ejecución de gasto/consumo de gobierno central de coyuntura (solo DIPRES) del mes t: analíticos predefinidos y variables, transferencias en especie de salud abiertas por ratio, ingresos operacionales imputados por entidades que informan agregado, e imputaciones permanentes incorporadas al total

*confianza: medium · verificador: approve · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, UNION ALL, DELETE/UPDATE y PROC DATASETS APPEND, encadenados sobre tablas WORK*

In [ ]:
# ========= OPC Salida CG_coy (solo DIP) =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# /*Crea tabla de apertura por actividad por PACACR ante la falta de C_ENTIDAD en EJEC_DIPRES */
# /*MODIFICADO JUNIO 2025*/ - /*Ejecucion dipres del periodo, con entidad asignada*/
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
    AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.Mes = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
# /*Analiticos predefinidos */
sql_ejecucion_a_prelim_dip = """
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS DEVENGO,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_a_prelim_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip))


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip (07/0)", res.rowcount)


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
# RATIOS ya existe en memoria (dependencia 4- Salida EJEC hasta CGR); se sube a #tmp para operar server-side
work_conn.execute(text("DROP TABLE IF EXISTS #ratios"))
ratios.to_sql("#ratios", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_d63"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.Ratio
INTO #ratios_d63
FROM #ratios t1
WHERE t1.C_SCN = 'D63'
"""))


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip2"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t1.[KEY],
    t2.Ratio
INTO #ejecucion_a_prelim_dip2
FROM #ejecucion_a_prelim_dip t1
LEFT JOIN #ratios_d63 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip2', '#ejecucion_a_prelim_dip'"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION,
    'IG' AS N_CUENTA, t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO * t1.Ratio AS DEVENGO, t1.[KEY]
INTO #ejecucion_a_prelim_dip_ajtesp
FROM #ejecucion_a_prelim_dip t1
WHERE t1.Ratio IS NOT NULL
"""))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip3"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO,
    t1.[KEY]
INTO #ejecucion_a_prelim_dip3
FROM #ejecucion_a_prelim_dip t1
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip3', '#ejecucion_a_prelim_dip'"))


In [ ]:
# PROC DATASETS APPEND BASE=EJECUCION_A_prelim_DIP DATA=EJECUCION_A_prelim_DIP_ajtesp FORCE
cols_ejecucion_a_prelim_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, "
    "DEVENGO, [KEY]"
)
sql_append = f"INSERT INTO #ejecucion_a_prelim_dip ({cols_ejecucion_a_prelim_dip}) SELECT {cols_ejecucion_a_prelim_dip} FROM #ejecucion_a_prelim_dip_ajtesp"
res = work_conn.execute(text(sql_append))
_log("APPEND #ejecucion_a_prelim_dip_ajtesp -> #ejecucion_a_prelim_dip", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_a_prelim_dip (15_04_01)", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip (32/analiticos)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("""
DELETE FROM #ejecucion_a_prelim_dip
WHERE [KEY] = '07_06_01' AND C_CUENTA = '06' AND C_ITEM = '01'
    AND C_ASIGNACION IN ('000','00') AND C_ANALITICO IN ('01000000')
"""))
_log("DELETE #ejecucion_a_prelim_dip (07_06_01 CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue). GENERA BASE AGREGADA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip
FROM #ejecucion_a_prelim_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""))


In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_b_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_b_dip (15_04_01)", res.rowcount)


In [ ]:
# /*ANALITICOS VARIABLES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_c_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_c_dip (15_04_01)", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
    t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_d_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
    AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t2.C_SCN NOT IN ('TC')
"""))


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.C_CUP AS C_ACT, t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_f_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
    AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""))


In [ ]:
# EJECUCION_G_DIP (DROP=ACTIVIDAD) desde EJEC_G_MIXT_DIP, que ya está en memoria (dependencia 5- TCons Mixtas_M_DIP_MOD)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip"))
ejec_g_mixt_dip.to_sql("#ejec_g_mixt_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.dato,
    CONCAT_WS('_', t1.MONEDA) AS [KEY]
INTO #ejecucion_g_dip
FROM #ejec_g_mixt_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL, t1.dato
"""))


In [ ]:
work_conn.execute(text("UPDATE #ejecucion_g_dip SET C_ACTIVO = 'TC'"))


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/ (DROP=KEY se aplica excluyendo la columna del SELECT)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_prev"))
cols_union_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_cg_dip_prev = f"""
SELECT {cols_union_dip} INTO #ejecucion_cg_dip_prev FROM #ejecucion_a_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_b_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_c_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_d_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_g_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_f_dip
"""
work_conn.execute(text(sql_cg_dip_prev))


In [ ]:
for t in ["#ejecucion_a_prelim_dip", "#ejecucion_a_dip", "#ejecucion_b_dip", "#ejecucion_c_dip", "#ejecucion_d_dip", "#ejecucion_g_dip", "#ejecucion_f_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("UPDATE #ejecucion_cg_dip_prev SET N_CUENTA = 'DIPRES'"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
          WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
          WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
          WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
          WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
          WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
          WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
          WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
          WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.Dato
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)}
    AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_imput_perm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip"))
work_conn.execute(text("""
SELECT * INTO #ejecucion_cg_dip FROM #ejecucion_cg_dip_prev
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imput_perm
"""))
ejecucion_cg_dip_imput_perm = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imput_perm"), work_conn)
_log("ejecucion_cg_dip_imput_perm", ejecucion_cg_dip_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""))


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
# COUNT()=1 filtra entidades con un único registro por esta llave; nota del analista: sin
# validación previa de duplicados, un cambio de granularidad en EJEC_DIPRES altera el filtro
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
    COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
    'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
    ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
    AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.id = 1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.C_ACTIVO,
    t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO
"""))


In [ ]:
# DATA EJECUCION_CG_DIP; SET EJECUCION_CG_DIP EJECUCION_DIP_07; -> concatenación server-side (append)
cols_cg_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_append_cg_dip = f"INSERT INTO #ejecucion_cg_dip ({cols_cg_dip}) SELECT {cols_cg_dip} FROM #ejecucion_dip_07"
res = work_conn.execute(text(sql_append_cg_dip))
_log("APPEND #ejecucion_dip_07 -> #ejecucion_cg_dip", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip WHERE NIVEL = 'EXCLUIDO'"))
_log("DELETE #ejecucion_cg_dip (EXCLUIDO)", res.rowcount)


In [ ]:
for t in ["#ejecucion_a_dip_07", "#ejecucion_dip_07", "#ejec_dip", "#cuenta_item_07", "#ejecucion_a_prelim_dip_ajtesp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
# Lecturas finales de control para dejar los datasets de salida en memoria, igual que el resto del notebook
ejecucion_cg_dip = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip"), work_conn)
_log("ejecucion_cg_dip", ejecucion_cg_dip)


## OPC Aper_CG_coy (solo DIP)

Ejecución desagregada de gasto de gobierno central (coyuntura) solo con datos DIPRES del mes vigente: clasifica analíticos SCN, abre transferencias en especie de salud pública según ratio, excluye entidades y analíticos ajustados por definición metodológica, agrega imputaciones permanentes e imputa ingresos operacionales de entidades que informan agregado

*confianza: low · verificador: revise · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, CASE WHEN, agregaciones, UNION ALL, DELETE, UPDATE y APPEND encadenados sobre datasets WORK derivados de tablas GOBGENER*

In [ ]:
# ========= OPC Aper_CG_coy (solo DIP) =========
# /*MODIFICADO JUNIO 2025*/ Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
  ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)


In [ ]:
# Analiticos predefinidos
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip_aper (07/0)", res.rowcount)


In [ ]:
# Estructura para abrir transferencias en especie... (RATIOS ya viene de un nodo anterior, dataframe en memoria)
ratios_d63 = ratios[ratios["C_SCN"] == "D63"][
    ["SUBSECTOR", "AÑO", "MONEDA", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
     "C_TIPO_CUENTA", "C_CUENTA", "C_SCN", "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION",
     "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT", "NIVEL", "Ratio"]
].copy()
ratios_d63.to_sql("#ratios_d63", work_conn, if_exists="replace", index=False)
_log("ratios_d63", ratios_d63)


In [ ]:
# apertura transferencias en especie de la salud pública
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper2"))
sql_ejecucion_a_prelim_dip_aper2 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t2.Ratio
INTO #ejecucion_a_prelim_dip_aper2
FROM #ejecucion_a_prelim_dip_aper t1
LEFT JOIN #ratios_d63 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper2))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip_aper2', '#ejecucion_a_prelim_dip_aper'"))


In [ ]:
# EJECUCION_A_prelim_DIP_ajtesp_ap: apertura fija D63/S14/433/transfesp/IG/272 cuando Ratio no es nulo
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp_ap"))
sql_ajtesp_ap = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_,
       'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION,
       'IG' AS N_CUENTA, t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO * t1.Ratio AS DEVENGO
INTO #ejecucion_a_prelim_dip_ajtesp_ap
FROM #ejecucion_a_prelim_dip_aper t1
WHERE t1.Ratio IS NOT NULL
"""
work_conn.execute(text(sql_ajtesp_ap))
# riesgo señalado por el analista: SAS no hace DROP de columnas temporales (ANALITICO_, C_ACT) en esta tabla; se conserva igual
ejecucion_a_prelim_dip_ajtesp_ap = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_ajtesp_ap"), work_conn)
_log("ejecucion_a_prelim_dip_ajtesp_ap", ejecucion_a_prelim_dip_ajtesp_ap)


In [ ]:
# recalcula DEVENGO aplicando (1-Ratio) donde Ratio no es nulo, deja DEVENGO igual si Ratio es nulo
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper2"))
sql_recalculo = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper2
FROM #ejecucion_a_prelim_dip_aper t1
"""
work_conn.execute(text(sql_recalculo))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip_aper2', '#ejecucion_a_prelim_dip_aper'"))


In [ ]:
# PROC DATASETS APPEND: agrega la apertura de transf. en especie a la tabla base, server-side
cols_ajtesp_ap = ", ".join([
    "SUBSECTOR", "MONEDA", "AÑO", "TRIM", "MES", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
    "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "ANALITICO_",
    "C_SCN", "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT",
    "NIVEL", "DEVENGO",
])
sql_append_ajtesp = f"INSERT INTO #ejecucion_a_prelim_dip_aper ({cols_ajtesp_ap}) SELECT {cols_ajtesp_ap} FROM #ejecucion_a_prelim_dip_ajtesp_ap"
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND #ejecucion_a_prelim_dip_aper (ajtesp_ap)", res.rowcount)
# fin apertura de transf en especie de salud pública


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_a_prelim_dip_aper (10201)", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (32/analiticos)", res.rowcount)


In [ ]:
# Analiticos predefinidos: AJUSTE COYUNTURA - se elimina indicador transacción CORFO que se desagrega anualmente entre ventas y D45
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION IN ('00','000') AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (3801 CORFO)", res.rowcount)


In [ ]:
# Analiticos predefinidos (sigue). BASE DESAGREGADA -- agrega por combinación de columnas del GROUP BY
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
         t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper))


In [ ]:
# Analiticos variables
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_b_dip_aper (10201)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_c_dip_aper (10201)", res.rowcount)


In [ ]:
# Transacciones diferenciadas para coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))


In [ ]:
# Transacciones diferenciadas por actividad
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT, t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))


In [ ]:
# EJECUCION_G_DIP_APER (DROP=ACTIVIDAD) -- se usa el ejec_g_mixt_aper_dip que dejó el nodo de TCons Mixtas_M_DIP_MOD
ejec_g_mixt_aper_dip.to_sql("#ejec_g_mixt_aper_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_ejecucion_g_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       SUM(t1.dato) AS Dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
         t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper))


In [ ]:
# UNE TABLAS de Salida total de ejecución CG coy de data DIPRES -- UNION ALL sin ORDER BY: orden no determinista (riesgo señalado por el analista)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_union = """
SELECT * INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL
SELECT * FROM #ejecucion_b_dip_aper
UNION ALL
SELECT * FROM #ejecucion_c_dip_aper
UNION ALL
SELECT * FROM #ejecucion_d_dip_aper
UNION ALL
SELECT * FROM #ejecucion_g_dip_aper
UNION ALL
SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_union))
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper", "#ejecucion_c_dip_aper", "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev (N_CUENTA=DIPRES)", res.rowcount)
# Se materializa: WORK.EJECUCION_CG_DIP_APER_PREV es salida del nodo
ejecucion_cg_dip_aper_prev = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_prev"), work_conn)
_log("ejecucion_cg_dip_aper_prev", ejecucion_cg_dip_aper_prev)


In [ ]:
# Incorporacion de imputaciones permanentes -- GOBGENER.IMPUT_GC_DIP_APER es tabla de BD (la deja el nodo OPC Agregar IMPUT a permanentes)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
             WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
             WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
             WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
             WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
             WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
             WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)} AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_imput_perm))
ejec_cg_dip_aper_imput_perm = pd.read_sql(text("SELECT * FROM #ejec_cg_dip_aper_imput_perm"), work_conn)
_log("ejec_cg_dip_aper_imput_perm", ejec_cg_dip_aper_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_ejecucion_cg_dip_aper = """
SELECT * INTO #ejecucion_cg_dip_aper FROM #ejecucion_cg_dip_aper_prev
UNION ALL
SELECT * FROM #ejec_cg_dip_aper_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper))


In [ ]:
# NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
sql_ejecucion_a_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_a_dip_07))
ejecucion_a_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_07"), work_conn)
_log("ejecucion_a_dip_07", ejecucion_a_dip_07)


In [ ]:
# Verifica que entidades informan solo el agregado para filtrarlas
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
       COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""
work_conn.execute(text(sql_cuenta_item_07))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_ejecucion_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
  ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
 AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA
 AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.id = 1
"""
work_conn.execute(text(sql_ejecucion_dip_07))
ejecucion_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_dip_07"), work_conn)
_log("ejecucion_dip_07", ejecucion_dip_07)


In [ ]:
# data EJECUCION_CG_DIP_APER; set EJECUCION_CG_DIP_APER EJECUCION_DIP_07; -- una imputacion de ingresos operacionales
cols_cg_dip_aper = ", ".join([
    "SUBSECTOR", "MONEDA", "AÑO", "TRIM", "MES", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
    "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "C_SCN",
    "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT", "NIVEL", "Dato",
])
sql_append_dip_07 = f"INSERT INTO #ejecucion_cg_dip_aper ({cols_cg_dip_aper}) SELECT {cols_cg_dip_aper} FROM #ejecucion_dip_07"
res = work_conn.execute(text(sql_append_dip_07))
_log("APPEND #ejecucion_cg_dip_aper (dip_07)", res.rowcount)
ejecucion_cg_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper"), work_conn)
_log("ejecucion_cg_dip_aper", ejecucion_cg_dip_aper)


In [ ]:
# limpieza final de tablas temporales de sesión, equivalente al PROC SQL DROP TABLE del SAS
for _t in ["#ejec_cg_dip_aper_imput_perm", "#cuenta_item_07", "#ejecucion_a_dip_07", "#ejecucion_a_prelim_dip_ajtesp_ap", "#ejec_dip", "#ejecucion_dip_07", "#ejecucion_cg_dip_aper_prev", "#ejecucion_cg_dip_aper", "#ratios_d63"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC Salida Ent CG_coy DIP

Ejecución de gasto de gobierno central de consumo (coyuntura), fuente DIPRES, del mes t, desagregada por analíticos predefinidos, variables, transacciones diferenciadas por coyuntura/actividad y mixtas, filtrada por el paquete de aperturas PACAPR y con imputaciones permanentes incorporadas

*confianza: medium · verificador: revise · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, CASE WHEN, GROUP BY, UNION ALL y DELETE/UPDATE sobre tablas de trabajo temporales*

In [ ]:
# ========= OPC Salida Ent CG_coy DIP =========
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr_v"))
# /*Crea tabla de apertura por actividad por PACACR ante la falta de C_ENTIDAD en EJEC_DIPRES */
sql_cr18_apertura_pacapr_v = """
SELECT DISTINCT t1.ACTIVIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.NIVEL
INTO #cr18_apertura_pacapr_v
FROM GOBGENER.dbo.CR18_ENTID_DIP t1
"""
work_conn.execute(text(sql_cr18_apertura_pacapr_v))
cr18_apertura_pacapr_v = pd.read_sql(text("SELECT * FROM #cr18_apertura_pacapr_v"), work_conn)
_log("cr18_apertura_pacapr_v", cr18_apertura_pacapr_v)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
# /*Analiticos predefinidos */
# CATX de SAS omite componentes nulos/en blanco sin agregar separador de más;
# CONCAT_WS de SQL Server solo omite NULL, no '' -- se normaliza a NULL con
# NULLIF para que el KEY resultante coincida con el de SAS y no altere el
# filtro KEY IN (&PACAPR1..&PACAPR7)
sql_ejecucion_a_prelim_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO='M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_a_prelim_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip))


In [ ]:
# proc sql; delete from EJECUCION_A_prelim_DIP where c_cuenta='07' and c_item='0'; quit; /*nuevo*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip (c_cuenta=07,c_item=0)", res.rowcount)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_a_prelim_dip (KEY=15_04_01)", res.rowcount)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip (c_cuenta=32)", res.rowcount)
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE KEY = '07_06_01' AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip (KEY=07_06_01 ajuste CORFO)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip"))
# /*Analiticos predefinidos (sigue)*/
sql_ejecucion_a_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       t1.C_ACT,
       t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip
FROM #ejecucion_a_prelim_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip))
ejecucion_a_dip = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip"), work_conn)
_log("ejecucion_a_dip", ejecucion_a_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip"))
# /*Analiticos variables */
sql_ejecucion_b_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_b_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_b_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_b_dip (KEY=15_04_01)", res.rowcount)
ejecucion_b_dip = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip"), work_conn)
_log("ejecucion_b_dip", ejecucion_b_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip"))
sql_ejecucion_c_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_c_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_c_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_c_dip (KEY=15_04_01)", res.rowcount)
ejecucion_c_dip = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip"), work_conn)
_log("ejecucion_c_dip", ejecucion_c_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip"))
# /* Transacciones diferenciadas para coyuntura */
sql_ejecucion_d_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_d_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_d_dip))
ejecucion_d_dip = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip"), work_conn)
_log("ejecucion_d_dip", ejecucion_d_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip"))
# /* Transacciones diferenciadas por actividad */
sql_ejecucion_f_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_f_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_f_dip))
ejecucion_f_dip = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip"), work_conn)
_log("ejecucion_f_dip", ejecucion_f_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip"))
# CREATE TABLE EJECUCION_G_DIP (DROP=ACTIVIDAD) -- KEY acá es CATX de un solo
# componente (MONEDA), que siempre viene poblado: CONCAT_WS('_', MONEDA) es
# equivalente sin riesgo de blancos
sql_ejecucion_g_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.DATO,
       CONCAT_WS('_', t1.MONEDA) AS KEY
INTO #ejecucion_g_dip
FROM #ejec_g_mixt_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', t1.MONEDA) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL, t1.DATO
"""
work_conn.execute(text(sql_ejecucion_g_dip))
res = work_conn.execute(text("UPDATE #ejecucion_g_dip SET C_ACTIVO = 'TC'"))
_log("UPDATE #ejecucion_g_dip C_ACTIVO=TC", res.rowcount)
ejecucion_g_dip = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip"), work_conn)
_log("ejecucion_g_dip", ejecucion_g_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_prev"))
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
# CREATE TABLE ... (DROP=KEY): KEY era solo auxiliar de filtro, no se incluye
sql_ejecucion_cg_dip_prev = """
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
INTO #ejecucion_cg_dip_prev
FROM #ejecucion_a_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_b_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_c_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_d_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_g_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_f_dip
"""
work_conn.execute(text(sql_ejecucion_cg_dip_prev))
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_prev N_CUENTA=DIPRES", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE ... QUIT; -- limpieza de tablas intermedias del SAS;
# se replica dropeando las #tmp correspondientes (CR18_APERTURA_PACAPR_v
# quedaba comentado en el DROP original: se conserva viva, la usan bloques
# posteriores del notebook)
for _t in ["#ejecucion_a_prelim_dip", "#ejecucion_a_dip", "#ejecucion_b_dip", "#ejecucion_c_dip", "#ejecucion_d_dip", "#ejecucion_g_dip", "#ejecucion_f_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
# /*Incorporacion de imputaciones permanentes*/
# GOBGENER.IMPUT_GC_DIP.KEY ya viene precalculado en la tabla de BD (no se
# recalcula acá, es columna existente)
sql_ejecucion_cg_dip_imput_perm = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN t1.C_ACT='FFAA' THEN 'AAPP'
             WHEN t1.C_ACT='FFOO' THEN 'AAPP'
             WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT='FONASA' THEN 'AAPP'
             WHEN t1.C_ACT='CORFO' THEN 'AAPP'
             WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT='AAPP' THEN 'AAPP'
             WHEN t1.C_ACT='SSPP' THEN 'SSPP'
             WHEN t1.C_ACT='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.Dato
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.MONEDA = '{Moneda}'
  AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_cg_dip_imput_perm))
ejecucion_cg_dip_imput_perm = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imput_perm"), work_conn)
_log("ejecucion_cg_dip_imput_perm", ejecucion_cg_dip_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip"))
sql_ejecucion_cg_dip = """
SELECT * INTO #ejecucion_cg_dip FROM #ejecucion_cg_dip_prev
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip))
ejecucion_cg_dip = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip"), work_conn)
_log("ejecucion_cg_dip", ejecucion_cg_dip)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))


## OPC Aper Ent CG_coy DIP

Ejecución coyuntural de Gobierno Central (CGR+DIPRES) desagregada por entidad y moneda específica, con analíticos predefinidos depurados, transacciones diferenciadas por coyuntura/actividad y las imputaciones permanentes incorporadas al cierre del periodo

*confianza: medium · verificador: approve · SAS: PROC SQL: múltiples CREATE TABLE con JOIN/CASE/GROUP BY, DELETE de analíticos predefinidos, UNION ALL de bloques y UPDATE, todo sobre tablas temporales de sesión*

In [ ]:
# ========= OPC Aper Ent CG_coy DIP =========
# Analíticos predefinidos: apertura por ACTIVIDAD/PARTIDA/CAPITULO/PROGRAMA a nivel DIP (bloque comentado del SAS ya no se usa)
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
sql_cr18_apertura_pacapr = """
SELECT DISTINCT t1.ACTIVIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.NIVEL
INTO #cr18_apertura_pacapr
FROM GOBGENER.dbo.CR18_ENTID_DIP t1
"""
work_conn.execute(text(sql_cr18_apertura_pacapr))
cr18_apertura_pacapr = pd.read_sql(text("SELECT * FROM #cr18_apertura_pacapr"), work_conn)
_log("cr18_apertura_pacapr", cr18_apertura_pacapr)


In [ ]:
# Analíticos predefinidos: EJECUCION_A_prelim_DIP_aper -> filtra cuentas fijas, cruza clasificación SCN_1, promedio mensual y apertura
# C_ENTIDAD se arma con INPUT(COMPRESS(CATX('0',partida,capitulo,programa), "1234567890.E-", "KIS"), E32.):
# COMPRESS con modificadores K/I/S invierte la lista (deja SOLO esos caracteres) e ignora mayúsc/minúsc y espacios;
# es decir, se queda con los dígitos/puntos/E/- de la concatenación y castea a número
cols_ej_a_prelim = (
    "SUBSECTOR, MONEDA, AÑO, "
    "CASE WHEN MES IN (1,2,3) THEN 1 WHEN MES IN (4,5,6) THEN 2 WHEN MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM, "
    "MES, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# proc sql; delete ... where c_cuenta='07' and c_item='0'; /*nuevo*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip_aper (c_cuenta=07,c_item=0)", res.rowcount)


In [ ]:
# DELETE ... WHERE C_ENTIDAD IN (15004001)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_a_prelim_dip_aper (C_ENTIDAD 15004001)", res.rowcount)


In [ ]:
# DELETE ... WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (C_CUENTA=32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_ENTIDAD IN (7006001) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (CORFO)", res.rowcount)


In [ ]:
# Analíticos predefinidos (sigue): EJECUCION_A_DIP_aper -> agrega DEVENGO por la clave de agrupación
# (SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL)
# el resto de columnas del SELECT (C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_ASIGNACION, C_ANALITICO, N_ANALITICO)
# no son función del GROUP BY: PROC SQL toma el valor del primer registro del grupo -> se replica con .first()
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES,
       MIN(C_ENTIDAD) AS C_ENTIDAD,
       MIN(C_PARTIDA) AS C_PARTIDA,
       MIN(C_CAPITULO) AS C_CAPITULO,
       MIN(C_PROGRAMA) AS C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM,
       MIN(C_ASIGNACION) AS C_ASIGNACION,
       MIN(C_ANALITICO) AS C_ANALITICO,
       MIN(N_ANALITICO) AS N_ANALITICO,
       C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL,
       SUM(DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper
GROUP BY SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
         C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper))
ejecucion_a_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_aper"), work_conn)
_log("ejecucion_a_dip_aper", ejecucion_a_dip_aper)


In [ ]:
# Analíticos variables: EJECUCION_B_DIP_aper (cruce con CR18_T_SCN_2, filtro C_CUENTA='32' y analíticos 04001000/04002000)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_b_dip_aper (C_ENTIDAD 15004001)", res.rowcount)
ejecucion_b_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_aper"), work_conn)
_log("ejecucion_b_dip_aper", ejecucion_b_dip_aper)


In [ ]:
# EJECUCION_C_DIP_aper: cuentas de deuda/depreciación (05,13,24,33), excluye OBS de ajuste coyuntura/actividad y SCN='TC'
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_c_dip_aper (C_ENTIDAD 15004001)", res.rowcount)
ejecucion_c_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_aper"), work_conn)
_log("ejecucion_c_dip_aper", ejecucion_c_dip_aper)


In [ ]:
# Transacciones diferenciadas para coyuntura: EJECUCION_D_DIP_aper (OBS='CR18_difcoy', C_SCN toma el valor de C_ACTIVO)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.ENTIDADES t5 ON t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO AND t1.C_PROGRAMA = t5.C_PROGRAMA
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))
ejecucion_d_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_aper"), work_conn)
_log("ejecucion_d_dip_aper", ejecucion_d_dip_aper)


In [ ]:
# Transacciones diferenciadas por actividad: EJECUCION_F_DIP_aper (OBS='CR18_difact', C_ACT toma el valor de C_CUP)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.ENTIDADES t5 ON t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO AND t1.C_PROGRAMA = t5.C_PROGRAMA
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))
ejecucion_f_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_aper"), work_conn)
_log("ejecucion_f_dip_aper", ejecucion_f_dip_aper)


In [ ]:
# EJECUCION_G_DIP_aper: agregación desde EJEC_G_MIXT_aper_DIP (dejado por CodeTask-99VPAhpFklGwBLLO en memoria)
# ACTIVIDAD se descarta del resultado (DROP=ACTIVIDAD del SAS)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip"))
ejec_g_mixt_aper_dip.to_sql("#ejec_g_mixt_aper_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_ejecucion_g_dip_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       CASE WHEN t1.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t1.NIVEL,
       SUM(t1.dato) AS dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
         t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper))
ejecucion_g_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_aper"), work_conn)
_log("ejecucion_g_dip_aper", ejecucion_g_dip_aper)


In [ ]:
# UNE TABLAS de Salida total de ejecución CG coy de data DIPRES
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_ejecucion_cg_dip_aper_prev = """
SELECT * INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL SELECT * FROM #ejecucion_b_dip_aper
UNION ALL SELECT * FROM #ejecucion_c_dip_aper
UNION ALL SELECT * FROM #ejecucion_d_dip_aper
UNION ALL SELECT * FROM #ejecucion_g_dip_aper
UNION ALL SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper_prev))


In [ ]:
# DROP TABLE EJECUCION_A_PRELIM_DIP_APER,EJECUCION_A_DIP_aper, EJECUCION_B_DIP_aper, EJECUCION_C_DIP_aper,
# EJECUCION_D_DIP_aper, EJECUCION_G_DIP_aper, EJECUCION_F_DIP_aper (limpieza de #tmp intermedias)
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper", "#ejecucion_c_dip_aper",
           "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# UPDATE EJECUCION_CG_DIP_aper_prev SET N_CUENTA='DIPRES'
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev (N_CUENTA=DIPRES)", res.rowcount)
ejecucion_cg_dip_aper_prev = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_prev"), work_conn)
_log("ejecucion_cg_dip_aper_prev", ejecucion_cg_dip_aper_prev)


In [ ]:
# Incorporación de imputaciones permanentes: EJEC_CG_DIP_aper_IMPUT_PERM desde GOBGENER.IMPUT_GC_DIP_APER (dejada por CodeTask-iUHB7DjImHdoG82f)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_ejec_cg_dip_aper_imput_perm = f"""
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP,
       CASE WHEN C_ACT IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN C_ACT = 'SSPP' THEN 'SSPP'
            WHEN C_ACT = 'EEPP' THEN 'EEPP' END AS C_ACT,
       NIVEL,
       Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER
WHERE AÑO = {int(Year)} AND C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND MONEDA = '{Moneda}' AND KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejec_cg_dip_aper_imput_perm))
ejec_cg_dip_aper_imput_perm = pd.read_sql(text("SELECT * FROM #ejec_cg_dip_aper_imput_perm"), work_conn)
_log("ejec_cg_dip_aper_imput_perm", ejec_cg_dip_aper_imput_perm)


In [ ]:
# EJECUCION_CG_DIP_aper: unión final de la salida coyuntural DIPRES con las imputaciones permanentes
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_ejecucion_cg_dip_aper = """
SELECT * INTO #ejecucion_cg_dip_aper FROM #ejecucion_cg_dip_aper_prev
UNION ALL SELECT * FROM #ejec_cg_dip_aper_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper))
ejecucion_cg_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper"), work_conn)
_log("ejecucion_cg_dip_aper", ejecucion_cg_dip_aper)


In [ ]:
# DROP TABLE EJEC_CG_DIP_aper_IMPUT_PERM; DROP TABLE CR18_APERTURA_PACAPR;
# (EJEC_CG_DIP_APER_IMPUT_PERM se mantiene como #tmp para nodos posteriores del flujo: no se dropea acá pese a que el SAS lo hacía,
# porque otras dependencias del notebook la consumen por su nombre real ejec_cg_dip_aper_imput_perm)
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
